In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8")
%matplotlib inline

CSV_PATH = Path("benchmark_results.csv")
MATRICES_PATH = Path("last_run_matrices.json")

In [ ]:
expected_columns = [
    "matrix_size",
    "serial_time_ms",
    "parallel_time_ms",
    "speedup",
    "num_servers",
    "timestamp",
]

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Arquivo nao encontrado: {CSV_PATH.resolve()}"
    )

df = pd.read_csv(CSV_PATH)

missing = set(expected_columns) - set(df.columns)

if missing:
    raise ValueError(
        f"O CSV nao possui as colunas esperadas: {sorted(missing)}"
    )

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce",
)

df = df.sort_values(
    [
        "matrix_size",
        "timestamp",
    ]
).reset_index(drop=True)

df


In [ ]:
summary = (
    df.groupby(
        [
            "matrix_size",
            "num_servers",
        ],
        as_index=False,
    )
    .agg(
        serial_time_ms=("serial_time_ms", "mean"),
        parallel_time_ms=("parallel_time_ms", "mean"),
        runs=("matrix_size", "count"),
    )
    .sort_values("matrix_size")
)

summary["speedup"] = (
    summary["serial_time_ms"] / summary["parallel_time_ms"]
)

summary


In [ ]:
if summary.empty:

    print("Sem dados para plotar.")

else:

    labels = [
        f"{int(matrix_size)}x{int(matrix_size)}"
        for matrix_size in summary["matrix_size"]
    ]

    serial_values = summary["serial_time_ms"].to_numpy()
    parallel_values = summary["parallel_time_ms"].to_numpy()
    speedup_values = summary["speedup"].to_numpy()

    # -----------------------------
    # Grafico de tempo
    # -----------------------------

    fig, ax = plt.subplots(figsize=(12, 6))

    ax.plot(
        labels,
        serial_values,
        marker="o",
        label="Serial",
    )

    ax.plot(
        labels,
        parallel_values,
        marker="o",
        label="Distribuido/paralelo",
    )

    crossing_candidates = np.where(parallel_values <= serial_values)[0]

    if len(crossing_candidates) > 0:
        turn_index = int(crossing_candidates[0])
    else:
        turn_index = int(np.argmin(np.abs(serial_values - parallel_values)))

    turn_y = max(serial_values[turn_index], parallel_values[turn_index])

    ax.annotate(
        "Ponto de virada",
        xy=(turn_index, turn_y),
        xytext=(turn_index, turn_y * 1.15 if turn_y > 0 else 1),
        arrowprops={"arrowstyle": "->", "color": "black"},
        ha="center",
    )

    ax.set_title(
        "Tempo de execucao por tamanho de matriz"
    )

    ax.set_xlabel("Tamanho da matriz")

    ax.set_ylabel("Tempo medio (ms)")

    ax.legend()

    ax.grid(True, alpha=0.3)

    plt.xticks(rotation=15)

    plt.show()

    # -----------------------------
    # Grafico de speedup
    # -----------------------------

    fig, ax = plt.subplots(figsize=(12, 6))

    bar_colors = [
        "#2e7d32" if speedup >= 1.0 else "#c62828"
        for speedup in speedup_values
    ]

    bars = ax.bar(
        labels,
        speedup_values,
        color=bar_colors,
    )

    ax.axhline(
        1,
        color="black",
        linewidth=1,
        linestyle="--",
        label="Linha de equilibrio (serial = distribuido)",
    )

    for bar, speedup in zip(bars, speedup_values):
        ax.annotate(
            f"{speedup:.2f}x",
            xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
            va="bottom",
        )

    ax.set_title(
        "Speedup da execucao distribuida"
    )

    ax.set_xlabel("Tamanho da matriz")

    ax.set_ylabel(
        "Speedup = tempo serial / tempo distribuido"
    )

    ax.grid(True, axis="y", alpha=0.3)
    ax.legend()

    plt.xticks(rotation=15)

    plt.show()


In [ ]:
labels = [
    f"{int(matrix_size)}x{int(matrix_size)}"
    for matrix_size in summary["matrix_size"]
]


In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
    labels,
    summary["serial_time_ms"],
    marker="o",
    linewidth=2,
    label="Serial",
)

plt.plot(
    labels,
    summary["parallel_time_ms"],
    marker="o",
    linewidth=2,
    label="Distribuido",
)

serial_values = summary["serial_time_ms"].to_numpy()
parallel_values = summary["parallel_time_ms"].to_numpy()

crossing_candidates = np.where(parallel_values <= serial_values)[0]

if len(crossing_candidates) > 0:
    turn_index = int(crossing_candidates[0])
else:
    turn_index = int(np.argmin(np.abs(serial_values - parallel_values)))

turn_y = max(serial_values[turn_index], parallel_values[turn_index])

plt.annotate(
    "Ponto de virada",
    xy=(turn_index, turn_y),
    xytext=(turn_index, turn_y * 1.15 if turn_y > 0 else 1),
    arrowprops={"arrowstyle": "->", "color": "black"},
    ha="center",
)

plt.xlabel("Tamanho da matriz")
plt.ylabel("Tempo (ms)")
plt.title("Serial vs Distribuido")

plt.xticks(rotation=15)

plt.grid(True)
plt.legend()

plt.show()


In [ ]:
def preview_matrix(matrix_data, rows=5, cols=5):

    matrix = np.array(matrix_data)

    return matrix[:rows, :cols]


if not MATRICES_PATH.exists():

    print(
        "Arquivo last_run_matrices.json nao encontrado. "
        "Execute o Client.py primeiro."
    )

else:

    with MATRICES_PATH.open(
        "r",
        encoding="utf-8",
    ) as json_file:

        last_run = json.load(json_file)

    print(
        f"Ultima execucao: "
        f"{last_run.get('timestamp', 'sem timestamp')}"
    )

    print(
        f"Servidores: "
        f"{', '.join(last_run.get('servers', []))}\n"
    )

    matrices = last_run.get("matrices", {})

    for matrix_key in matrices:

        item = matrices[matrix_key]

        matrix_size = item["matrix_size"]

        print(f"Matriz {matrix_size}x{matrix_size}")

        print()

        print("  Matriz A (primeiras 5x5):")

        print(
            preview_matrix(item["A"])
        )

        print()

        print("  Matriz B (primeiras 5x5):")

        print(
            preview_matrix(item["B"])
        )

        print()

        for partial in item.get(
            "partial_results",
            [],
        ):

            server_number = partial.get(
                "server",
                "?",
            )

            rows = partial.get(
                "rows",
                "?",
            )

            cols = partial.get(
                "cols",
                "?",
            )

            print(
                f"  Resultado parcial do "
                f"Servidor {server_number} "
                f"({rows}x{cols}) "
                f"- primeiras 5x5:"
            )

            print(
                preview_matrix(
                    partial["matrix"]
                )
            )

            print()

        print(
            "  Matriz C final "
            "(primeiras 5x5):"
        )

        print(
            preview_matrix(item["C"])
        )

        print()


In [ ]:
if summary.empty:

    print(
        "Ainda nao ha resultados para analisar."
    )

else:

    for row in summary.itertuples(index=False):

        print(
            f"Matriz {int(row.matrix_size)}x{int(row.matrix_size)} "
            f"com "
            f"{int(row.num_servers)} servidores: "
            f"serial={row.serial_time_ms:.2f} ms, "
            f"distribuido={row.parallel_time_ms:.2f} ms, "
            f"speedup={row.speedup:.2f}x "
            f"({int(row.runs)} execucao/execucoes)."
        )

    best = summary.loc[
        summary["speedup"].idxmax()
    ]

    print()

    print(
        "Melhor speedup observado: "
        f"{best['speedup']:.2f}x "
        f"para "
        f"matriz {int(best['matrix_size'])}x{int(best['matrix_size'])}."
    )
